# 02 · Validación del pipeline

1. Conteos de cada tabla de Bronze, Silver y Gold (los de Bronze/Silver van al README después de cada lote).
2. Registros rechazados por fuente (`silver.cuarentena`, política `drop`) y advertencias (`warn`).
3. **Falla** si alguna tabla de Gold queda vacía.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo del ambiente")
CAT = dbutils.widgets.get("catalogo")

def tablas(schema):
    return [r.tableName for r in spark.sql(f"SHOW TABLES IN {CAT}.{schema}").collect() if not r.isTemporary]

conteos = []
for capa in ["bronze", "silver", "gold"]:
    for t in sorted(tablas(capa)):
        n = spark.table(f"{CAT}.{capa}.{t}").count()
        conteos.append((capa, t, n))
df_conteos = spark.createDataFrame(conteos, "capa STRING, tabla STRING, filas BIGINT")
display(df_conteos)

### Bronze por lote / archivo (evidencia de que las reingestas no duplican en Silver)

In [ ]:
for t in ["ventas", "devoluciones", "empleados_eventos", "resenas", "catalogo", "sucursales"]:
    display(spark.sql(f"SELECT '{t}' AS tabla, _lote, _source_file, count(*) AS filas "
                      f"FROM {CAT}.bronze.{t} GROUP BY ALL ORDER BY _source_file"))
display(spark.sql(f"""
SELECT 'ventas' AS fuente,
       (SELECT count(*) FROM {CAT}.bronze.ventas)                      AS filas_bronze,
       (SELECT count(DISTINCT venta_id) FROM {CAT}.bronze.ventas)      AS ids_distintos_bronze,
       (SELECT count(*) FROM {CAT}.silver.ventas)                      AS filas_silver
UNION ALL
SELECT 'devoluciones',
       (SELECT count(*) FROM {CAT}.bronze.devoluciones),
       (SELECT count(DISTINCT devolucion_id) FROM {CAT}.bronze.devoluciones),
       (SELECT count(*) FROM {CAT}.silver.devoluciones)
UNION ALL
SELECT 'resenas',
       (SELECT count(*) FROM {CAT}.bronze.resenas),
       (SELECT count(DISTINCT resena_id) FROM {CAT}.bronze.resenas),
       (SELECT count(*) FROM {CAT}.silver.resenas)
UNION ALL
SELECT 'tracking',
       (SELECT count(*) FROM {CAT}.bronze.tracking_envios),
       (SELECT count(DISTINCT evento_id) FROM {CAT}.bronze.tracking_envios),
       (SELECT count(*) FROM {CAT}.silver.tracking)
"""))

### Registros rechazados por fuente

In [ ]:
display(spark.sql(f"""
SELECT fuente,
       count(DISTINCT CASE WHEN politica = 'drop' THEN coalesce(id_registro, registro) END) AS registros_rechazados,
       count(DISTINCT CASE WHEN politica = 'warn' THEN coalesce(id_registro, registro) END) AS registros_con_advertencia
FROM {CAT}.silver.cuarentena
GROUP BY fuente
ORDER BY fuente
"""))
display(spark.sql(f"""
SELECT fuente, regla, politica, count(*) AS violaciones
FROM {CAT}.silver.cuarentena
GROUP BY ALL
ORDER BY fuente, politica, violaciones DESC
"""))

### Gold no puede quedar vacío

In [ ]:
GOLD_OBLIGATORIAS = [
    "ventas_netas_sucursal_mes", "top_productos_unidades", "top_productos_tasa_devolucion",
    "dotacion_sucursal_mes", "ventas_vendedor_sucursal_mes", "resenas_negativas_categoria_mes",
    "despachos_courier",
]
existentes = set(tablas("gold"))
vacias = []
for t in GOLD_OBLIGATORIAS:
    if t not in existentes:
        vacias.append(f"{t} (no existe)")
    elif spark.table(f"{CAT}.gold.{t}").limit(1).count() == 0:
        vacias.append(t)

if vacias:
    raise Exception(f"Validación fallida: tablas de Gold vacías o inexistentes -> {vacias}")
print("Validación OK: todas las tablas de Gold tienen datos.")